<a href="https://colab.research.google.com/github/alexanderconroy/literary-summarization-llms/blob/main/literary_summarization_llms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports and setup

In [ ]:
# imports

!pip install openai
!pip install together
!pip install --pre fireworks-ai
!pip install tenacity
!pip install krippendorff

import os, json, time
from tenacity import retry, wait_random_exponential, stop_after_attempt
from openai import OpenAI
from together import Together
from fireworks import Fireworks
import pandas as pd
import numpy as np
import krippendorff
import re
from sklearn.metrics import cohen_kappa_score
from itertools import combinations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.7/415.7 kB 26.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# data

input_dir = "/content/drive/MyDrive/[my path]/novels"
output_dir = "/content/drive/MyDrive/[my path]/summaries"
log_path = os.path.join(output_dir, "runs.jsonl")
os.makedirs(output_dir, exist_ok=True)



In [ ]:
# API's
openai_api_key = ""
together_api_key = ""
fireworks_api_key = ""

openai_client = OpenAI(api_key=openai_api_key)
together_client = Together(api_key=together_api_key)
fireworks_client = Fireworks(api_key=fireworks_api_key, timeout=180.0,)


In [ ]:
# setup

temperature = 0.2
top_p = 1.0

# for hierarchical merging & sequential
words_per_chunk = 12000
words_overlap = 1200

# token limits for the strategies and their steps

max_full = 6000
max_seq_step  = 1600
max_seq_final = 2000
max_chunk = 800
max_merge = 2000
max_biblio = 2000

# functions used

def read_text(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def write_text(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def log_row(d):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

def list_novel_files():
    return sorted([f for f in os.listdir(input_dir) if f.endswith(".txt")])

def sentence_aligned_chunks(text, words_per_chunk, words_overlap):
    words = text.split()
    n = len(words)
    chunks = []
    start = 0

    while start < n:
        end = min(start + words_per_chunk, n)
        while end < n and not words[end - 1].endswith('.'):
            end += 1

        chunks.append(" ".join(words[start:end]))

        if end >= n:
            break

        start = max(0, end - words_overlap)
        while start > 0 and not words[start - 1].endswith('.'):
            start += 1

    return chunks


@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6), reraise=True)
def call_llm(provider, model, messages, max_tokens):

    if provider == "openai":
        r = openai_client.responses.create(
            model=model,
            input=messages,
            reasoning={"effort": "low"},
            max_output_tokens=max_tokens,
        )

        text = (r.output_text or "").strip()

        if not text:
            raise RuntimeError(
                f"Empty output_text. usage={getattr(r, 'usage', None)}"
        )

        return text

    if provider == "together":
        r = together_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
        )
        content = r.choices[0].message.content
        if not content or not content.strip():
            raise RuntimeError("Empty content from Together API")
        return content.strip()

    if provider == "fireworks":
        r = fireworks_client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
        )
        content = r.choices[0].message.content
        if not content or not content.strip():
            raise RuntimeError("Empty content from Fireworks API")
        return content.strip()

    raise ValueError(provider)

In [ ]:
# checking the length of the books

for fn in list_novel_files():
    book = os.path.splitext(fn)[0]
    text = read_text(os.path.join(input_dir, fn))

    word_count = len(text.split())
    approx_tokens = int(word_count * 1.4)

    print(f"{book}")
    print(f"  Words: {word_count:,}")
    print(f"  Approx tokens: {approx_tokens:,}")
    print()

1870_AT_DenVanvittigesBoernEllerSkurkenPaaNoerrebro.paragraphed.txt_ocrfixed.paragraphed
  Words: 74,216
  Approx tokens: 103,902

1872_LeFevre_DianaEllerHaabloesKjaerlighed.paragraphed.txt_ocrfixed.paragraphed
  Words: 73,728
  Approx tokens: 103,219

1876_JacobsenJP_FruMarieGrubbe.paragraphed.txt_ocrfixed.paragraphed
  Words: 75,523
  Approx tokens: 105,732

1880_JacobsenJP_NielsLyhne.paragraphed.txt_ocrfixed.paragraphed
  Words: 66,822
  Approx tokens: 93,550

1881_Aniken_Pavo.paragraphed.txt_ocrfixed.paragraphed
  Words: 62,389
  Approx tokens: 87,344

1886_Winterhjelm_Naturalisterne.paragraphed.txt_ocrfixed.paragraphed
  Words: 30,857
  Approx tokens: 43,199

1889_BangH_Tine.paragraphed.txt_ocrfixed.paragraphed
  Words: 46,670
  Approx tokens: 65,337

1890_Hamsun_Sult.paragraphed.txt_ocrfixed.paragraphed
  Words: 60,185
  Approx tokens: 84,259

1896_LundV_Praesten.paragraphed.txt_ocrfixed.paragraphed
  Words: 34,998
  Approx tokens: 48,997

1897_Janson_Mira.paragraphed.txt_ocrfixe

# Full novel summarized

In [ ]:
# prompts
system_prompt = """You are a careful and attentive summarizer. Your task is to produce faithful summaries that strictly reflect the content of the source text, written in an objective and academically appropriate tone for literary-historical research. Do not invent or add any information that is not explicitly stated or clearly supported by the text. Do not make up names of people, places, events, or other details. If something is unclear or not specified, omit it rather than guessing. Focus on getting the content factually right rather than interpreting the novel."""
user_prompt = """Please provide a Danish summary of the following novel. The summary must not exceed 400 words. The summary should mention the fictional time period or historical setting (if relevant), the key places or locations within the novel, the main characters, and the major events and developments of the plot in the correct narrative order. Do include themes, but keep interpretation to an absolute minimum. Only include themes that are clearly grounded in the narrative, not vague or generic abstractions or clichés. Write the summary as a continuous and readable text without bullet points or other structured formatting."""

In [ ]:
# gpt-5.2
provider = "openai"
model = "gpt-5.2"

for fn in list_novel_files():

    book = os.path.splitext(fn)[0]
    text = read_text(os.path.join(input_dir, fn))

    outdir = os.path.join(output_dir, f"{book}__{provider}__{model.replace('/','_')}", "fulltext")
    os.makedirs(outdir, exist_ok=True)

    messages = [
        {"role":"system", "content": system_prompt},
        {"role":"user", "content": f"{user_prompt}\n\n{text}"}
    ]

    t0 = time.time()
    summary = call_llm(provider, model, messages, max_full)
    dt = time.time() - t0

    task = "full_text"
    model_for_filename = model.replace("/", "_")
    out_path = os.path.join(outdir, f"{book}__{task}__{model_for_filename}.txt")
    write_text(out_path, summary)

    log_row({"task":"full_text", "book":book, "provider":provider, "model":model, "temperature": temperature,
  "top_p": top_p, "max_full": max_full, "seconds": dt, "out":out_path,})

print("done", provider, model)

done openai gpt-5.2


In [ ]:
# gpt-5 mini
provider = "openai"
model = "gpt-5-mini"

for fn in list_novel_files():

    book = os.path.splitext(fn)[0]
    text = read_text(os.path.join(input_dir, fn))

    outdir = os.path.join(output_dir, f"{book}__{provider}__{model.replace('/','_')}", "fulltext")
    os.makedirs(outdir, exist_ok=True)

    messages = [
        {"role":"system", "content": system_prompt},
        {"role":"user", "content": f"{user_prompt}\n\n{text}"}
    ]

    t0 = time.time()
    summary = call_llm(provider, model, messages, max_full)
    dt = time.time() - t0

    task = "full_text"
    model_for_filename = model.replace("/", "_")
    out_path = os.path.join(outdir, f"{book}__{task}__{model_for_filename}.txt")
    write_text(out_path, summary)

    log_row({"task":"full_text", "book":book, "provider":provider, "model":model, "temperature": temperature,
  "top_p": top_p, "max_full": max_full, "seconds": dt, "out":out_path,})

print("done", provider, model)

done openai gpt-5-mini


In [ ]:
# kimi k2
provider = "together"
model = "moonshotai/Kimi-K2-Instruct-0905"

for fn in list_novel_files():

    book = os.path.splitext(fn)[0]
    text = read_text(os.path.join(input_dir, fn))

    outdir = os.path.join(output_dir, f"{book}__{provider}__{model.replace('/','_')}", "fulltext")
    os
.makedirs(outdir, exist_ok=True)

    messages = [
        {"role":"system", "content": system_prompt},
        {"role":"user", "content": f"{user_prompt}\n\n{text}"}
    ]

    t0 = time.time()
    summary = call_llm(provider, model, messages, max_full)
    dt = time.time() - t0

    task = "full_text"
    model_for_filename = model.replace("/", "_")
    out_path = os.path.join(outdir, f"{book}__{task}__{model_for_filename}.txt")
    write_text(out_path, summary)

    log_row({"task":"full_text", "book":book, "provider":provider, "model":model, "temperature": temperature,
  "top_p": top_p, "max_full": max_full, "seconds": dt, "out":out_path,})

print("done", provider, model)

done together moonshotai/Kimi-K2-Thinking


# Hierarchical

In [ ]:
# prompts

system_prompt = """You are a careful and attentive summarizer. Your task is to produce faithful summaries that strictly reflect the content of the source text, written in an objective and academically appropriate tone for literary-historical research. Do not invent or add any information that is not explicitly stated or clearly supported by the text. Do not make up names of people, places, events, or other details. If something is unclear or not specified, omit it rather than guessing. Focus on getting the content factually right rather than interpreting the novel."""
chunk_prompt = """Please provide a Danish summary of the following PART of a novel. The summary should mention the fictional time period or historical setting (if relevant within this part), the key places or locations that appear in this part, the main characters present in this part, and the major events and developments that occur here, in the correct narrative order. Do include themes, but keep interpretation to an absolute minimum. Only include themes that are clearly grounded in the narrative of this part, not vague or generic abstractions or clichés. Write the summary as a continuous and readable text without bullet points or other structured formatting."""
merge_prompt = """Please provide a Danish summary of the following novel. The summary must not exceed 400 words. The text below consists of summaries of parts of the novel. The summary should mention the fictional time period or historical setting (if relevant), the key places or locations within the novel, the main characters, and the major events and developments of the plot in the correct narrative order. Do include themes, but keep interpretation to an absolute minimum. Only include themes that are clearly grounded in the narrative, not vague or generic abstractions or clichés. Write the summary as a continuous and readable text without bullet points or other structured formatting. Compress the final summary so that it does not exceed 400 words. If compression is necessary, shorten wording across the whole summary rather than deleting earlier major events."""

In [ ]:
# gpt 5.2

provider = "openai"
model = "gpt-5.2"

chunk_system = system_prompt

chunk_user = chunk_prompt

merge_system = system_prompt
merge_user = merge_prompt

for fn in list_novel_files():
    t0 = time.time()
    book = os.path.splitext(fn)[0]
    text = read_text(os.path.join(input_dir, fn))

    outdir = os.path.join(output_dir, f"{book}__{provider}__{model.replace('/','_')}", "hier_24k_o2k")
    os.makedirs(outdir, exist_ok=True)
    os.makedirs(os.path.join(outdir, "chunks"), exist_ok=True)

    chunks = sentence_aligned_chunks(text, words_per_chunk, words_overlap)

    chunk_summaries = []
    for i, ch in enumerate(chunks):
        p = os.path.join(outdir, "chunks", f"sum_{i:03d}.txt")
        if os.path.exists(p):
            chunk_summaries.append(read_text(p))
            continue

        messages = [
            {"role":"system", "content": chunk_system},
            {"role":"user", "content": f"{chunk_user}\n\n{ch}"}
        ]

        s = call_llm(provider, model, messages, max_chunk)
        write_text(p, s)
        chunk_summaries.append(s)

    merged_input = "\n\n".join(chunk_summaries)

    merge_messages = [
        {"role":"system", "content": merge_system},
        {"role":"user", "content": f"{merge_user}\n\n{merged_input}"}
    ]

    summary = call_llm(provider, model, merge_messages, max_merge)
    task = "hier_24k"
    model_for_filename = model.replace("/", "_")
    filename = f"{book}__{task}__{model_for_filename}.txt"
    out_path = os.path.join(outdir, filename)
    write_text(out_path, summary)
    dt = time.time() - t0

    log_row({"task":"hier_24k", "book":book, "provider":provider, "model":model, "temperature": temperature,
  "top_p": top_p, "max_chunk": max_chunk, "max_merge": max_merge,
      "words_per_chunk": words_per_chunk,"words_overlap": words_overlap, "seconds": dt,
      "out":out_path, "n_chunks":len(chunks)})

print("done", provider, model)

done openai gpt-5.2


In [ ]:
# gpt 5 mini

provider = "openai"
model = "gpt-5-mini"

chunk_system = system_prompt

chunk_user = chunk_prompt

merge_system = system_prompt
merge_user = merge_prompt

for fn in list_novel_files():
    t0 = time.time()
    book = os.path.splitext(fn)[0]
    text = read_text(os.path.join(input_dir, fn))

    outdir = os.path.join(output_dir, f"{book}__{provider}__{model.replace('/','_')}", "hier_24k_o2k")
    os.makedirs(outdir, exist_ok=True)
    os.makedirs(os.path.join(outdir, "chunks"), exist_ok=True)

    chunks = sentence_aligned_chunks(text, words_per_chunk, words_overlap)

    chunk_summaries = []
    for i, ch in enumerate(chunks):
        p = os.path.join(outdir, "chunks", f"sum_{i:03d}.txt")
        if os.path.exists(p):
            chunk_summaries.append(read_text(p))
            continue

        messages = [
            {"role":"system", "content": chunk_system},
            {"role":"user", "content": f"{chunk_user}\n\n{ch}"}
        ]

        s = call_llm(provider, model, messages, max_chunk)
        write_text(p, s)
        chunk_summaries.append(s)

    merged_input = "\n\n".join(chunk_summaries)

    merge_messages = [
        {"role":"system", "content": merge_system},
        {"role":"user", "content": f"{merge_user}\n\n{merged_input}"}
    ]

    summary = call_llm(provider, model, merge_messages, max_merge)
    task = "hier_24k"
    model_for_filename = model.replace("/", "_")
    filename = f"{book}__{task}__{model_for_filename}.txt"
    out_path = os.path.join(outdir, filename)
    write_text(out_path, summary)
    dt = time.time() - t0

    log_row({"task":"hier_24k", "book":book, "provider":provider, "model":model, "temperature": temperature,
  "top_p": top_p, "max_chunk": max_chunk, "max_merge": max_merge,
      "words_per_chunk": words_per_chunk,"words_overlap": words_overlap, "seconds": dt,
      "out":out_path, "n_chunks":len(chunks)})

print("done", provider, model)

done openai gpt-5-mini


In [ ]:
# Kimi
provider = "together"
model = "moonshotai/Kimi-K2-Instruct-0905"

chunk_system = system_prompt

chunk_user = chunk_prompt

merge_system = system_prompt
merge_user = merge_prompt

for fn in list_novel_files():
    t0 = time.time()
    book = os.path.splitext(fn)[0]
    text = read_text(os.path.join(input_dir, fn))

    outdir = os.path.join(output_dir, f"{book}__{provider}__{model.replace('/','_')}", "hier_24k_o2k")
    os.makedirs(outdir, exist_ok=True)
    os.makedirs(os.path.join(outdir, "chunks"), exist_ok=True)

    chunks = sentence_aligned_chunks(text, words_per_chunk, words_overlap)

    chunk_summaries = []
    for i, ch in enumerate(chunks):
        p = os.path.join(outdir, "chunks", f"sum_{i:03d}.txt")
        if os.path.exists(p):
            chunk_summaries.append(read_text(p))
            continue

        messages = [
            {"role":"system", "content": chunk_system},
            {"role":"user", "content": f"{chunk_user}\n\n{ch}"}
        ]

        s = call_llm(provider, model, messages, max_chunk)
        write_text(p, s)
        chunk_summaries.append(s)

    merged_input = "\n\n".join(chunk_summaries)

    merge_messages = [
        {"role":"system", "content": merge_system},
        {"role":"user", "content": f"{merge_user}\n\n{merged_input}"}
    ]

    summary = call_llm(provider, model, merge_messages, max_merge)
    task = "hier_24k"
    model_for_filename = model.replace("/", "_")
    filename = f"{book}__{task}__{model_for_filename}.txt"
    out_path = os.path.join(outdir, filename)
    write_text(out_path, summary)
    dt = time.time() - t0

    log_row({"task":"hier_24k", "book":book, "provider":provider, "model":model, "temperature": temperature,
  "top_p": top_p, "max_chunk": max_chunk, "max_merge": max_merge,
      "words_per_chunk": words_per_chunk,"words_overlap": words_overlap, "seconds": dt,
      "out":out_path, "n_chunks":len(chunks)})

print("done", provider, model)

# Metadata

In [ ]:
# prompts

system_prompt_biblio = """You are a careful and attentive summarizer. Your task is to produce faithful summaries written in an objective and academically appropriate tone for literary-historical research. Do not invent or add information that is clearly incorrect. If something is unclear or not specified, omit it rather than guessing. Focus on getting the content factually right rather than interpreting the novel."""

user_prompt_biblio = """Please provide a Danish summary of the following novel. The summary must not exceed 400 words. The summary should mention the fictional time period or historical setting (if relevant), the key places or locations within the novel, the main characters, and the major events and developments of the plot in the correct narrative order. Do include themes, but keep interpretation to an absolute minimum. Only include themes that are clearly grounded in the novel, not vague or generic abstractions or clichés. Write the summary as a continuous and readable text without bullet points or other structured formatting."""


In [ ]:
BIBLIO = {
    "den_vanvittiges_børn": {
        "title": "Den Vanvittiges Børn eller Skurken paa Nørrebro",
        "subtitle": "Original romantisk Fortælling",
        "author": "A.T.",
        "year": 1870
    },
    "diana": {
        "title": "Diana eller Haabløs Kjærlighed",
        "subtitle": "Original romantisk Fortælling",
        "author": "P. B. le Fevre",
        "year": 1872
    },
    "fru_marie_grubbe": {
        "title": "Fru Marie Grubbe",
        "subtitle": "Interieurer fra det syttende Aarhundrede",
        "author": "J.P. Jacobsen",
        "year": 1876
    },
    "niels_lyhne": {
        "title": "Niels Lyhne",
        "subtitle": "Roman",
        "author": "J.P. Jacobsen",
        "year": 1880
    },
    "pavo": {
        "title": "Pavo",
        "subtitle": "Original romantisk Fortælling",
        "author": "Aniken",
        "year": 1881
    },
    "naturalisterne": {
        "title": "Naturalisterne",
        "subtitle": "En Sommerskitse fra Stockholms Skjærgaard",
        "author": "Kristian Winterhjelm",
        "year": 1886
    },
    "tine": {
        "title": "Tine",
        "subtitle": "",
        "author": "Herman Bang",
        "year": 1889
    },
    "sult": {
        "title": "Sult",
        "subtitle": "",
        "author": "Knut Hamsun",
        "year": 1890
    },
    "præsten": {
        "title": "Præsten",
        "subtitle": "Fortælling",
        "author": "Viggo Lund",
        "year": 1896
    },
    "Mira": {
        "title": "Mira",
        "subtitle": "Et Livsløb",
        "author": "Drude Krog Janson",
        "year": 1897
    },
  }

In [ ]:
provider = "openai"
model = "gpt-5.2"

model_for_filename = model.replace("/", "_")

for book, biblio in BIBLIO.items():
    task = "metadata"

    outdir = os.path.join(
        output_dir,
        f"{book}__{provider}__{model_for_filename}",
        task
    )
    os.makedirs(outdir, exist_ok=True)

    messages = [
        {"role": "system", "content": system_prompt_biblio},
        {"role": "user", "content": f"{user_prompt_biblio}\n\n{json.dumps(biblio, ensure_ascii=False)}"}
    ]

    summary = call_llm(provider, model, messages, max_biblio)

    out_path = os.path.join(outdir, f"{book}__{task}__{model_for_filename}.txt")
    write_text(out_path, summary)

    log_row({
        "task": task,
        "book": book,
        "provider": provider,
        "model": model,
        "out": out_path,
        "biblio_record": biblio
    })

print("done", provider, model)

done openai gpt-5.2


In [ ]:
provider = "openai"
model = "gpt-5-mini"

model_for_filename = model.replace("/", "_")

for book, biblio in BIBLIO.items():
    task = "metadata"

    outdir = os.path.join(
        output_dir,
        f"{book}__{provider}__{model_for_filename}",
        task
    )
    os.makedirs(outdir, exist_ok=True)

    messages = [
        {"role": "system", "content": system_prompt_biblio},
        {"role": "user", "content": f"{user_prompt_biblio}\n\n{json.dumps(biblio, ensure_ascii=False)}"}
    ]

    summary = call_llm(provider, model, messages, max_biblio)

    out_path = os.path.join(outdir, f"{book}__{task}__{model_for_filename}.txt")
    write_text(out_path, summary)

    log_row({
        "task": task,
        "book": book,
        "provider": provider,
        "model": model,
        "out": out_path,
        "biblio_record": biblio
    })

print("done", provider, model)

done openai gpt-5-mini


In [ ]:
provider = "together"
model = "moonshotai/Kimi-K2-Instruct-0905"

model_for_filename = model.replace("/", "_")

for book, biblio in BIBLIO.items():
    task = "metadata"

    outdir = os.path.join(
        output_dir,
        f"{book}__{provider}__{model_for_filename}",
        task
    )
    os.makedirs(outdir, exist_ok=True)

    messages = [
        {"role": "system", "content": system_prompt_biblio},
        {"role": "user", "content": f"{user_prompt_biblio}\n\n{json.dumps(biblio, ensure_ascii=False)}"}
    ]

    summary = call_llm(provider, model, messages, max_biblio)

    out_path = os.path.join(outdir, f"{book}__{task}__{model_for_filename}.txt")
    write_text(out_path, summary)

    log_row({
        "task": task,
        "book": book,
        "provider": provider,
        "model": model,
        "out": out_path,
        "biblio_record": biblio
    })

print("done", provider, model)

done together deepseek-ai/DeepSeek-V3


# Sequential

**Note on terminology:** This strategy is referred to as incremental updating in the paper. The name difference is purely terminological. This is the code used for that strategy.

**Note on pipeline:** This pipeline was added after the initial setup, which is reflected in the separate annotation loading below.


**Note on Kimi:** The provider for Kimi was switched mid-run, as the original provider had deprecated the model between runs.


In [ ]:
# prompts
system_prompt = """You are a careful and attentive summarizer. Your task is to produce faithful summaries that strictly reflect the content of the source text, written in an objective and academically appropriate tone for literary-historical research. Do not invent or add any information that is not explicitly stated or clearly supported by the text. Do not make up names of people, places, events, or other details. If something is unclear or not specified, omit it rather than guessing. Focus on getting the content factually right rather than interpreting the novel."""
seq_step_prompt  = "Please provide a Danish summary of the novel. The novel is presented in consecutive chunks, and you must maintain a cumulative summary up to the current point in the narrative. You are given: (1) an existing summary of the novel so far and (2) a new PART of the novel. Update the summary so that it covers everything from the beginning up to and including this new part. Revise the summary as a whole so that it remains balanced across the narrative so far. You may compress or rephrase earlier parts to make room for new developments. Do not produce a standalone summary of the new chunk. Instead, return only the revised cumulative summary. For the first chunk, no existing summary will be provided. In that case, write a new summary covering that part only. The summary should mention the fictional time period or historical setting (if relevant), the key places or locations that appear so far, the main characters present so far, and the major events and developments from the beginning up to this point, in the correct narrative order. Do include themes, but keep interpretation to an absolute minimum. Only include themes that are clearly grounded in the narrative so far, not vague or generic abstractions or clichés. Update the summary so that it covers everything from the beginning up to and including this new part. Write the summary as a continuous and readable text without bullet points or other structured formatting. Keep the running summary concise and avoid unnecessary detail, as it will later be compressed."
# final polish prompt
seq_final_prompt = "Please provide a Danish summary of the following novel. The text below consists of a cumulative summary of the novel that was built incrementally from consecutive parts. The summary should mention the fictional time period or historical setting (if relevant), the key places or locations within the novel, the main characters, and the major events and developments of the plot in the correct narrative order. Do include themes, but keep interpretation to an absolute minimum. Only include themes that are clearly grounded in the narrative, not vague or generic abstractions or clichés. Write the summary as a continuous and readable text without bullet points or other structured formatting. Do not introduce new information. Only refine and consolidate the provided summary. Compress the final summary so that it does not exceed 400 words. If compression is necessary, shorten and rebalance the summary across the entire narrative, preserving the most important events from all parts of the novel."

In [ ]:
provider = "openai"
model = "gpt-5.2"

seq_system = system_prompt

for fn in list_novel_files():

    t0 = time.time()
    book = os.path.splitext(fn)[0]
    text = read_text(os.path.join(input_dir, fn))

    outdir = os.path.join(
        output_dir,
        f"{book}__{provider}__{model.replace('/','_')}",
        "sequential"
    )
    os.makedirs(outdir, exist_ok=True)
    os.makedirs(os.path.join(outdir, "steps"), exist_ok=True)

    chunks = sentence_aligned_chunks(text, words_per_chunk, words_overlap)


    state_path = os.path.join(outdir, "state_latest.txt")
    state = ""

    for i, ch in enumerate(chunks):

        step_path = os.path.join(outdir, "steps", f"step_{i:03d}.txt")

        messages = [
            {"role":"system", "content": seq_system},
            {"role":"user", "content":
                f"{seq_step_prompt}\n\n"
                f"=== RUNNING SUMMARY SO FAR (EDIT THIS TEXT) ===\n{state}\n\n"
                f"=== NEXT TEXT CHUNK (SOURCE ONLY) ===\n{ch}\n\n"
                f"Return ONLY the edited RUNNING SUMMARY SO FAR."
            }
        ]

        state = call_llm(provider, model, messages, max_seq_step)

        write_text(step_path, state)
        write_text(state_path, state)


    final_messages = [
        {"role":"system", "content": seq_system},
        {"role":"user", "content":
            f"{seq_final_prompt}\n\n"
            f"=== ACCUMULATED SUMMARY ===\n{state}"
        }
    ]

    final_summary = call_llm(provider, model, final_messages, max_seq_final)

    out_path = os.path.join(
        outdir,
        f"{book}__sequential__{model.replace('/','_')}.txt"
    )
    write_text(out_path, final_summary)

    dt = time.time() - t0

    log_row({
        "task":"sequential",
        "book":book,
        "provider":provider,
        "model":model,
        "temperature":temperature,
        "top_p":top_p,
        "max_seq_step":max_seq_step,
        "max_seq_final":max_seq_final,
        "words_per_chunk":words_per_chunk,
        "words_overlap":words_overlap,
        "seconds":dt,
        "out":out_path,
        "n_chunks":len(chunks),
    })

print("done", provider, model)

done openai gpt-5.2


In [ ]:
provider = "openai"
model = "gpt-5-mini"

seq_system = system_prompt

for fn in list_novel_files():

    t0 = time.time()
    book = os.path.splitext(fn)[0]
    text = read_text(os.path.join(input_dir, fn))

    outdir = os.path.join(
        output_dir,
        f"{book}__{provider}__{model.replace('/','_')}",
        "sequential"
    )
    os.makedirs(outdir, exist_ok=True)
    os.makedirs(os.path.join(outdir, "steps"), exist_ok=True)

    chunks = sentence_aligned_chunks(text, words_per_chunk, words_overlap)


    state_path = os.path.join(outdir, "state_latest.txt")
    state = ""

    for i, ch in enumerate(chunks):

        step_path = os.path.join(outdir, "steps", f"step_{i:03d}.txt")

        messages = [
            {"role":"system", "content": seq_system},
            {"role":"user", "content":
                f"{seq_step_prompt}\n\n"
                f"=== RUNNING SUMMARY SO FAR (EDIT THIS TEXT) ===\n{state}\n\n"
                f"=== NEXT TEXT CHUNK (SOURCE ONLY) ===\n{ch}\n\n"
                f"Return ONLY the edited RUNNING SUMMARY SO FAR."
            }
        ]

        state = call_llm(provider, model, messages, max_seq_step)

        write_text(step_path, state)
        write_text(state_path, state)


    final_messages = [
        {"role":"system", "content": seq_system},
        {"role":"user", "content":
            f"{seq_final_prompt}\n\n"
            f"=== ACCUMULATED SUMMARY ===\n{state}"
        }
    ]

    final_summary = call_llm(provider, model, final_messages, max_seq_final)

    out_path = os.path.join(
        outdir,
        f"{book}__sequential__{model.replace('/','_')}.txt"
    )
    write_text(out_path, final_summary)

    dt = time.time() - t0

    log_row({
        "task":"sequential",
        "book":book,
        "provider":provider,
        "model":model,
        "temperature":temperature,
        "top_p":top_p,
        "max_seq_step":max_seq_step,
        "max_seq_final":max_seq_final,
        "words_per_chunk":words_per_chunk,
        "words_overlap":words_overlap,
        "seconds":dt,
        "out":out_path,
        "n_chunks":len(chunks),
    })

print("done", provider, model)

done openai gpt-5-mini


In [ ]:
provider = "fireworks"
model = "accounts/fireworks/models/kimi-k2-instruct-0905"

seq_system = system_prompt

for fn in list_novel_files():

    t0 = time.time()
    book = os.path.splitext(fn)[0]
    text = read_text(os.path.join(input_dir, fn))

    outdir = os.path.join(
        output_dir,
        f"{book}__{provider}__{model.replace('/','_')}",
        "sequential"
    )
    os.makedirs(outdir, exist_ok=True)
    os.makedirs(os.path.join(outdir, "steps"), exist_ok=True)

    chunks = sentence_aligned_chunks(text, words_per_chunk, words_overlap)


    state_path = os.path.join(outdir, "state_latest.txt")
    state = ""

    for i, ch in enumerate(chunks):

        step_path = os.path.join(outdir, "steps", f"step_{i:03d}.txt")

        messages = [
            {"role":"system", "content": seq_system},
            {"role":"user", "content":
                f"{seq_step_prompt}\n\n"
                f"=== RUNNING SUMMARY SO FAR (EDIT THIS TEXT) ===\n{state}\n\n"
                f"=== NEXT TEXT CHUNK (SOURCE ONLY) ===\n{ch}\n\n"
                f"Return ONLY the edited RUNNING SUMMARY SO FAR."
            }
        ]

        state = call_llm(provider, model, messages, max_seq_step)

        write_text(step_path, state)
        write_text(state_path, state)


    final_messages = [
        {"role":"system", "content": seq_system},
        {"role":"user", "content":
            f"{seq_final_prompt}\n\n"
            f"=== ACCUMULATED SUMMARY ===\n{state}"
        }
    ]

    final_summary = call_llm(provider, model, final_messages, max_seq_final)

    out_path = os.path.join(
        outdir,
        f"{book}__sequential__{model.replace('/','_')}.txt"
    )
    write_text(out_path, final_summary)

    dt = time.time() - t0

    log_row({
        "task":"sequential",
        "book":book,
        "provider":provider,
        "model":model,
        "temperature":temperature,
        "top_p":top_p,
        "max_seq_step":max_seq_step,
        "max_seq_final":max_seq_final,
        "words_per_chunk":words_per_chunk,
        "words_overlap":words_overlap,
        "seconds":dt,
        "out":out_path,
        "n_chunks":len(chunks),
    })

print("done", provider, model)

done fireworks accounts/fireworks/models/kimi-k2-instruct-0905


# Data analysis

## Data preparation

In [ ]:
# read the csv and extract the data that was collected via google forms

raw_df = pd.read_csv(
    "/content/drive/MyDrive/[my path]/Annotation_summarization_Jens_transponeret.csv",
    header=None,
    names=["field", "value"],
    encoding="utf-8-sig"
)

parsed_rows = []
current_novel = None

for _, row in raw_df.iterrows():
    field = str(row["field"]).strip()
    value = row["value"]

    # find the novel id's
    if field == "novel_id":
        current_novel = str(value).strip()
        continue

    # find summary number
    summary_match = re.search(r"Summary\s*#\s*(\d+)", field, flags=re.IGNORECASE)

    # find the categories
    category_match = re.search(r"\[([^\]]+)\]\s*$", field, flags=re.DOTALL)

    if current_novel and summary_match and category_match:
        score = pd.to_numeric(value, errors="coerce")

        if pd.notna(score):
            parsed_rows.append({
                "novel_id": current_novel,
                "summary_number": int(summary_match.group(1)),
                "category": category_match.group(1).strip(),
                "score": float(score),
                "annotator": "Jens"
            })

# make the data frame
annotations_jens_df = pd.DataFrame(parsed_rows)

# check results
print(annotations_jens_df.head())
print(annotations_jens_df["novel_id"].unique())
print(annotations_jens_df.shape)

           novel_id  summary_number    category  score annotator
0  Fru Marie Grubbe               1     Fluency    3.0      Jens
1  Fru Marie Grubbe               1   Coherence    4.0      Jens
2  Fru Marie Grubbe               1   Relevance    4.0      Jens
3  Fru Marie Grubbe               1  Factuality    4.0      Jens
4  Fru Marie Grubbe               1        Time    5.0      Jens
['Fru Marie Grubbe' 'Niels Lyhne' 'Tine' 'Sult' 'Naturalisterne' 'Pavo']
(486, 5)


In [ ]:
# the same is done for the other annotator files
raw_df = pd.read_csv(
    "/content/drive/MyDrive/[my path]/Annotation_summarization_Kirstine_transponeret.csv",
    header=None,
    names=["field", "value"],
    encoding="utf-8-sig"
)

parsed_rows = []
current_novel = None

for _, row in raw_df.iterrows():
    field = str(row["field"]).strip()
    value = row["value"]

    if field == "novel_id":
        current_novel = str(value).strip()
        continue

    summary_match = re.search(r"Summary\s*#\s*(\d+)", field, flags=re.IGNORECASE)

    category_match = re.search(r"\[([^\]]+)\]\s*$", field, flags=re.DOTALL)

    if current_novel and summary_match and category_match:
        score = pd.to_numeric(value, errors="coerce")

        if pd.notna(score):
            parsed_rows.append({
                "novel_id": current_novel,
                "summary_number": int(summary_match.group(1)),
                "category": category_match.group(1).strip(),
                "score": float(score),
                "annotator": "Kirstine"
            })

annotations_kirstine_df = pd.DataFrame(parsed_rows)

In [ ]:
# and again

raw_df = pd.read_csv(
    "/content/drive/MyDrive/[my data]/Annotation_summarization_Alexander_transponeret.csv",
    header=None,
    names=["field", "value"],
    encoding="utf-8-sig"
)

parsed_rows = []
current_novel = None

for _, row in raw_df.iterrows():
    field = str(row["field"]).strip()
    value = row["value"]

    if field == "novel_id":
        current_novel = str(value).strip()
        continue

    summary_match = re.search(r"Summary\s*#\s*(\d+)", field, flags=re.IGNORECASE)

    category_match = re.search(r"\[([^\]]+)\]\s*$", field, flags=re.DOTALL)

    if current_novel and summary_match and category_match:
        score = pd.to_numeric(value, errors="coerce")

        if pd.notna(score):
            parsed_rows.append({
                "novel_id": current_novel,
                "summary_number": int(summary_match.group(1)),
                "category": category_match.group(1).strip(),
                "score": float(score),
                "annotator": "Alexander"
            })

annotations_alexander_df = pd.DataFrame(parsed_rows)

In [ ]:
# then they are combined into one df

combined_annotations_df = pd.concat(
    [annotations_jens_df, annotations_kirstine_df, annotations_alexander_df],
    ignore_index=True
)

In [ ]:
# novel id's are mapped to novel names
novel_to_number = {
    "Fru Marie Grubbe": 1,
    "Niels Lyhne": 2,
    "Tine": 3,
    "Sult": 4,
    "Naturalisterne": 5,
    "Pavo": 6,
    "Mira": 7,
    "Diana": 8,
    "Præsten": 9,
    "Skurken på Nørrebro": 10,
}

combined_annotations_df["novel_number"] = combined_annotations_df["novel_id"].map(novel_to_number)

# then the novel number is mapped to the order of summaries
group_1 = {  # Roman 1, 4, 7, 10
    1: ("GPT5.2", "Full_text"),
    2: ("GPT5.2", "Hierarchical"),
    3: ("GPT5.2", "Metadata"),
    4: ("GPT5-mini", "Full_text"),
    5: ("GPT5-mini", "Hierarchical"),
    6: ("GPT5-mini", "Metadata"),
    7: ("Kimi", "Full_text"),
    8: ("Kimi", "Hierarchical"),
    9: ("Kimi", "Metadata"),
}

group_2 = {  # Roman 2, 5, 8
    1: ("GPT5-mini", "Hierarchical"),
    2: ("GPT5-mini", "Metadata"),
    3: ("GPT5-mini", "Full_text"),
    4: ("Kimi", "Hierarchical"),
    5: ("Kimi", "Metadata"),
    6: ("Kimi", "Full_text"),
    7: ("GPT5.2", "Hierarchical"),
    8: ("GPT5.2", "Metadata"),
    9: ("GPT5.2", "Full_text"),
}

group_3 = {  # Roman 3, 6, 9
    1: ("Kimi", "Metadata"),
    2: ("Kimi", "Full_text"),
    3: ("Kimi", "Hierarchical"),
    4: ("GPT5.2", "Metadata"),
    5: ("GPT5.2", "Full_text"),
    6: ("GPT5.2", "Hierarchical"),
    7: ("GPT5-mini", "Metadata"),
    8: ("GPT5-mini", "Full_text"),
    9: ("GPT5-mini", "Hierarchical"),
}

In [ ]:
# mapping function
def get_model_strategy(novel_number, summary_number):
    if novel_number in [1, 4, 7, 10]:
        return group_1.get(summary_number, (None, None))
    elif novel_number in [2, 5, 8]:
        return group_2.get(summary_number, (None, None))
    elif novel_number in [3, 6, 9]:
        return group_3.get(summary_number, (None, None))
    return (None, None)

In [ ]:
combined_annotations_df["summary_number"] = pd.to_numeric(
    combined_annotations_df["summary_number"], errors="coerce"
).astype("Int64")

In [ ]:
# adding the columns to the df
combined_annotations_df[["model", "strategy"]] = combined_annotations_df.apply(
    lambda row: pd.Series(get_model_strategy(row["novel_number"], row["summary_number"])),
    axis=1
)

# check
print(combined_annotations_df.head())

           novel_id  summary_number    category  score annotator  \
0  Fru Marie Grubbe               1     Fluency    3.0      Jens   
1  Fru Marie Grubbe               1   Coherence    4.0      Jens   
2  Fru Marie Grubbe               1   Relevance    4.0      Jens   
3  Fru Marie Grubbe               1  Factuality    4.0      Jens   
4  Fru Marie Grubbe               1        Time    5.0      Jens   

   novel_number   model   strategy  
0             1  GPT5.2  Full_text  
1             1  GPT5.2  Full_text  
2             1  GPT5.2  Full_text  
3             1  GPT5.2  Full_text  
4             1  GPT5.2  Full_text  


**Adding the sequential annotations**

In [ ]:
raw_df = pd.read_csv(
    "/content/drive/MyDrive/[my path]/Annotation_summarization_Jens_sekventiel_transponeret.csv",
    header=None,
    names=["field", "value"],
    encoding="utf-8-sig"
)

parsed_rows = []
current_novel = None

for _, row in raw_df.iterrows():
    field = str(row["field"]).strip()
    value = row["value"]

    if field == "novel_id":
        current_novel = str(value).strip()
        continue

    summary_match = re.search(r"Summary\s*#\s*(\d+)", field, flags=re.IGNORECASE)

    category_match = re.search(r"\[([^\]]+)\]\s*$", field, flags=re.DOTALL)

    if current_novel and summary_match and category_match:
        score = pd.to_numeric(value, errors="coerce")

        if pd.notna(score):
            parsed_rows.append({
                "novel_id": current_novel,
                "summary_number": int(summary_match.group(1)),
                "category": category_match.group(1).strip(),
                "score": float(score),
                "annotator": "Jens"
            })

annotations_jens_df_sekventiel = pd.DataFrame(parsed_rows)

In [ ]:
raw_df = pd.read_csv(
    "/content/drive/MyDrive/[my path]/Annotation_summarization_Kirstine_sekventiel_transponeret.csv",
    header=None,
    names=["field", "value"],
    encoding="utf-8-sig"
)

parsed_rows = []
current_novel = None

for _, row in raw_df.iterrows():
    field = str(row["field"]).strip()
    value = row["value"]

    if field == "novel_id":
        current_novel = str(value).strip()
        continue

    summary_match = re.search(r"Summary\s*#\s*(\d+)", field, flags=re.IGNORECASE)

    category_match = re.search(r"\[([^\]]+)\]\s*$", field, flags=re.DOTALL)

    if current_novel and summary_match and category_match:
        score = pd.to_numeric(value, errors="coerce")

        if pd.notna(score):
            parsed_rows.append({
                "novel_id": current_novel,
                "summary_number": int(summary_match.group(1)),
                "category": category_match.group(1).strip(),
                "score": float(score),
                "annotator": "Kirstine"
            })

annotations_kirstine_df_sekventiel = pd.DataFrame(parsed_rows)

In [ ]:
raw_df = pd.read_csv(
    "/content/drive/MyDrive/[my path]/Annotation_summarization_Alexander_sekventiel_transponeret.csv",
    header=None,
    names=["field", "value"],
    encoding="utf-8-sig"
)

parsed_rows = []
current_novel = None

for _, row in raw_df.iterrows():
    field = str(row["field"]).strip()
    value = row["value"]

    if field == "novel_id":
        current_novel = str(value).strip()
        continue

    summary_match = re.search(r"Summary\s*#\s*(\d+)", field, flags=re.IGNORECASE)

    category_match = re.search(r"\[([^\]]+)\]\s*$", field, flags=re.DOTALL)

    if current_novel and summary_match and category_match:
        score = pd.to_numeric(value, errors="coerce")

        if pd.notna(score):
            parsed_rows.append({
                "novel_id": current_novel,
                "summary_number": int(summary_match.group(1)),
                "category": category_match.group(1).strip(),
                "score": float(score),
                "annotator": "Alexander"
            })

annotations_alexander_df_sekventiel = pd.DataFrame(parsed_rows)

In [ ]:
combined_annotations_df_sekventiel = pd.concat(
    [annotations_jens_df_sekventiel, annotations_kirstine_df_sekventiel, annotations_alexander_df_sekventiel],
    ignore_index=True
)

In [ ]:
novel_to_number = {
    "Fru Marie Grubbe": 1,
    "Niels Lyhne": 2,
    "Tine": 3,
    "Sult": 4,
    "Naturalisterne": 5,
    "Pavo": 6,
    "Mira": 7,
    "Diana": 8,
    "Præsten": 9,
    "Skurken på Nørrebro": 10,
}

combined_annotations_df_sekventiel["novel_number"] = (
    combined_annotations_df_sekventiel["novel_id"].map(novel_to_number)
)

group_1 = {  # Novel 1, 4, 7, 10
    1: "GPT5.2",
    2: "GPT5-mini",
    3: "Kimi",
}

group_2 = {  # Novel 2, 5, 8
    1: "Kimi",
    2: "GPT5.2",
    3: "GPT5-mini",
}

group_3 = {  # Novel 3, 6, 9
    1: "GPT5-mini",
    2: "Kimi",
    3: "GPT5.2",
}

In [ ]:
def get_model_strategy(novel_number, summary_number):
    if novel_number in [1, 4, 7, 10]:
        model = group_1.get(summary_number, None)
    elif novel_number in [2, 5, 8]:
        model = group_2.get(summary_number, None)
    elif novel_number in [3, 6, 9]:
        model = group_3.get(summary_number, None)
    else:
        model = None

    strategy = "Sequential" if model is not None else None
    return model, strategy


In [ ]:
combined_annotations_df_sekventiel["summary_number"] = pd.to_numeric(
    combined_annotations_df_sekventiel["summary_number"], errors="coerce"
).astype("Int64")

In [ ]:
combined_annotations_df_sekventiel[["model", "strategy"]] = (
    combined_annotations_df_sekventiel.apply(
        lambda row: pd.Series(
            get_model_strategy(row["novel_number"], row["summary_number"])
        ),
        axis=1
    )
)

In [ ]:
# concatenate the sequential df to make one dataset

combined_annotations_all_df = pd.concat(
    [combined_annotations_df, combined_annotations_df_sekventiel],
    ignore_index=True
)

In [ ]:
# check
combined_annotations_all_df

,novel_id,summary_number,category,score,annotator,novel_number,model,strategy
0,Fru Marie Grubbe,1,Fluency,3.0,Jens,1,GPT5.2,Full_text
1,Fru Marie Grubbe,1,Coherence,4.0,Jens,1,GPT5.2,Full_text
2,Fru Marie Grubbe,1,Relevance,4.0,Jens,1,GPT5.2,Full_text
3,Fru Marie Grubbe,1,Factuality,4.0,Jens,1,GPT5.2,Full_text
4,Fru Marie Grubbe,1,Time,5.0,Jens,1,GPT5.2,Full_text
...,...,...,...,...,...,...,...,...
1939,Skurken på Nørrebro,3,Time,4.0,Alexander,10,Kimi,Sequential
1940,Skurken på Nørrebro,3,Place,4.0,Alexander,10,Kimi,Sequential
1941,Skurken på Nørrebro,3,Characters,4.0,Alexander,10,Kimi,Sequential
1942,Skurken på Nørrebro,3,Plot,5.0,Alexander,10,Kimi,Sequential


## IAA

In [ ]:
# here the inter-annotator agreement is calculated using Krippendorff's alpha (ordinal)

df = combined_annotations_all_df.copy()

df["novel_number"] = pd.to_numeric(df["novel_number"], errors="coerce").astype("Int64")
df["summary_number"] = pd.to_numeric(df["summary_number"], errors="coerce").astype("Int64")
df["score"] = pd.to_numeric(df["score"], errors="coerce")

df_iaa = df[df["novel_number"].isin([1, 2, 3, 4])].copy()

df_iaa["unit_id"] = (
    df_iaa["novel_number"].astype(str)
    + "_"
    + df_iaa["summary_number"].astype(str)
    + "_"
    + df_iaa["category"].astype(str)
    + "_"
    + df_iaa["strategy"].astype(str)
)

# overall score
pivot_all = df_iaa.pivot_table(
    index="unit_id",
    columns="annotator",
    values="score",
    aggfunc="first"
).dropna(how="any")

alpha_all = krippendorff.alpha(
    reliability_data=pivot_all.T.to_numpy(),
    level_of_measurement="ordinal"
)

print("Samlet Krippendorff's alpha (ordinal), roman 1-4:", round(alpha_all, 4))

# and per category
results = []

for cat in sorted(df_iaa["category"].dropna().unique()):
    pivot_cat = (
        df_iaa[df_iaa["category"] == cat]
        .pivot_table(
            index="unit_id",
            columns="annotator",
            values="score",
            aggfunc="first"
        )
        .dropna(how="any")
    )

    if len(pivot_cat) > 0:
        alpha_cat = krippendorff.alpha(
            reliability_data=pivot_cat.T.to_numpy(),
            level_of_measurement="ordinal"
        )
    else:
        alpha_cat = float("nan")

    results.append({
        "category": cat,
        "n_units": len(pivot_cat),
        "krippendorff_alpha_ordinal": alpha_cat
    })

results_df = pd.DataFrame(results)
results_df["krippendorff_alpha_ordinal"] = results_df["krippendorff_alpha_ordinal"].round(4)

print(results_df)

Samlet Krippendorff's alpha (ordinal), roman 1-4: 0.5173
     category  n_units  krippendorff_alpha_ordinal
0  Characters       48                      0.5021
1   Coherence       48                      0.2735
2  Factuality       48                      0.5583
3     Fluency       48                      0.2627
4       Place       48                      0.5205
5        Plot       48                      0.5967
6   Relevance       48                      0.5289
7      Themes       48                      0.3579
8        Time       48                      0.6319


In [ ]:
# pairwise agreement between annotaters using Cohen's kappa

df = combined_annotations_all_df.copy()

df = df[df["novel_number"].isin([1,2,3,4])].copy()

df["unit_id"] = (
    df["novel_number"].astype(str)
    + "_"
    + df["summary_number"].astype(str)
    + "_"
    + df["category"]
    + "_"
    + df["strategy"].astype(str)
)

pivot = df.pivot_table(
    index="unit_id",
    columns="annotator",
    values="score",
    aggfunc="first"
)

annotators = pivot.columns.tolist()

results = []

for a1, a2 in combinations(annotators, 2):

    pair = pivot[[a1, a2]].dropna()

    kappa = cohen_kappa_score(
        pair[a1],
        pair[a2],
        weights="quadratic"
    )

    results.append({
        "annotator_1": a1,
        "annotator_2": a2,
        "n_units": len(pair),
        "weighted_kappa": kappa
    })

pairwise_kappa_df = pd.DataFrame(results)

print(pairwise_kappa_df)

  annotator_1 annotator_2  n_units  weighted_kappa
0   Alexander        Jens      432        0.629751
1   Alexander    Kirstine      432        0.622671
2        Jens    Kirstine      432        0.414707


In [ ]:
# and pairwise per category

df = combined_annotations_all_df.copy()

df = df[df["novel_number"].isin([1,2,3,4])].copy()

df["unit_id"] = (
    df["novel_number"].astype(str)
    + "_"
    + df["summary_number"].astype(str)
    + "_"
    + df["category"]
    + "_"
    + df_iaa["strategy"].astype(str)
)

results = []

categories = sorted(df["category"].unique())
annotators = sorted(df["annotator"].unique())

for cat in categories:

    df_cat = df[df["category"] == cat]

    pivot = df_cat.pivot_table(
        index="unit_id",
        columns="annotator",
        values="score",
        aggfunc="first"
    )

    for a1, a2 in combinations(annotators, 2):

        pair = pivot[[a1, a2]].dropna()

        if len(pair) > 0:

            kappa = cohen_kappa_score(
                pair[a1],
                pair[a2],
                weights="quadratic"
            )

            results.append({
                "category": cat,
                "annotator_1": a1,
                "annotator_2": a2,
                "n_units": len(pair),
                "weighted_kappa": kappa
            })

pairwise_category_df = pd.DataFrame(results)

print(pairwise_category_df)

      category annotator_1 annotator_2  n_units  weighted_kappa
0   Characters   Alexander        Jens       48        0.589977
1   Characters   Alexander    Kirstine       48        0.622989
2   Characters        Jens    Kirstine       48        0.388471
3    Coherence   Alexander        Jens       48        0.458333
4    Coherence   Alexander    Kirstine       48        0.285714
5    Coherence        Jens    Kirstine       48        0.109375
6   Factuality   Alexander        Jens       48        0.634286
7   Factuality   Alexander    Kirstine       48        0.768116
8   Factuality        Jens    Kirstine       48        0.547779
9      Fluency   Alexander        Jens       48        0.286174
10     Fluency   Alexander    Kirstine       48        0.317726
11     Fluency        Jens    Kirstine       48        0.253333
12       Place   Alexander        Jens       48        0.576159
13       Place   Alexander    Kirstine       48        0.473251
14       Place        Jens    Kirstine  

In [ ]:
# calculation of annotater mean difference, exact agreement, and how often disagreements are 1-point differences

pivot = combined_annotations_all_df[
    combined_annotations_all_df["novel_number"].isin([1,2,3,4])
].pivot_table(
    index=["novel_number", "strategy", "summary_number", "category"],
    columns="annotator",
    values="score"
)

pairs = [
    ("Alexander", "Jens"),
    ("Alexander", "Kirstine"),
    ("Jens", "Kirstine")
]

for a1, a2 in pairs:
    diff = (pivot[a1] - pivot[a2]).abs()

    print(a1, "vs", a2)
    print("mean difference:", diff.mean())
    print("exact agreement:", (diff == 0).mean())
    print("≤1 difference:", (diff <= 1).mean())
    print()

Alexander vs Jens
mean difference: 0.48842592592592593
exact agreement: 0.5578703703703703
≤1 difference: 0.9560185185185185

Alexander vs Kirstine
mean difference: 0.6134259259259259
exact agreement: 0.4837962962962963
≤1 difference: 0.9050925925925926

Jens vs Kirstine
mean difference: 0.8240740740740741
exact agreement: 0.3472222222222222
≤1 difference: 0.8472222222222222



In [ ]:
# exact agreement across all annotators

exact_all_three = (
    (pivot["Alexander"] == pivot["Jens"]) &
    (pivot["Alexander"] == pivot["Kirstine"])
).mean()

print("all three exact agreement:", exact_all_three)

all three exact agreement: 0.26157407407407407


In [ ]:
# how often disagreements are point differences for all annotators
within1_all_three = (
    pivot.max(axis=1) - pivot.min(axis=1) <= 1
).mean()

print("all three within-1 agreement:", within1_all_three)

all three within-1 agreement: 0.7939814814814815


In [ ]:
# calculating the 1-point differences for categories

pivot_iaa = combined_annotations_all_df[
    combined_annotations_all_df["novel_number"].isin([1, 2, 3, 4])
].pivot_table(
    index=["novel_number", "strategy", "summary_number", "category"],
    columns="annotator",
    values="score"
)

categories = sorted(pivot_iaa.index.get_level_values("category").unique())
annotator_pairs = list(combinations(["Alexander", "Jens", "Kirstine"], 2))

agreement_results = []

for cat in categories:
    cat_pivot = pivot_iaa.xs(cat, level="category").dropna()

    within1_all = (cat_pivot.max(axis=1) - cat_pivot.min(axis=1) <= 1).mean()

    cat_res = {"category": cat, "All Three (≤1)": round(within1_all, 3)}

    for a1, a2 in annotator_pairs:
        pair_within1 = (np.abs(cat_pivot[a1] - cat_pivot[a2]) <= 1).mean()
        cat_res[f"{a1} vs {a2} (≤1)"] = round(pair_within1, 3)

    agreement_results.append(cat_res)

within1_df = pd.DataFrame(agreement_results)

within1_df

,category,All Three (≤1),Alexander vs Jens (≤1),Alexander vs Kirstine (≤1),Jens vs Kirstine (≤1)
0,Characters,0.771,0.958,0.896,0.833
1,Coherence,0.812,0.958,0.938,0.854
2,Factuality,0.875,0.979,0.958,0.917
3,Fluency,0.750,0.938,0.854,0.896
4,Place,0.792,0.979,0.875,0.833
5,Plot,0.812,0.917,0.938,0.854
6,Relevance,0.792,0.958,0.896,0.833
7,Themes,0.750,0.958,0.917,0.771
8,Time,0.792,0.958,0.875,0.833


## Model performance

In [ ]:
# model performance across all strategies

df = combined_annotations_all_df.copy()

df["score"] = pd.to_numeric(df["score"], errors="coerce")

unit_scores = (
    df.groupby(
        ["novel_number", "summary_number", "model", "strategy", "category"]
    )["score"]
    .mean()
    .reset_index()
)

In [ ]:
model_category_performance = (
    unit_scores
    .groupby(["model", "category"])["score"]
    .mean()
    .reset_index()
)

In [ ]:
model_category_table = unit_scores.pivot_table(
    index="category",
    columns="model",
    values="score",
    aggfunc="mean"
).round(2)

In [ ]:
model_category_table.index.name = None
model_category_table.columns.name = None

display(
    model_category_table.style
    .format("{:.2f}")
    .highlight_max(axis=1, props="text-decoration: underline; text-decoration-thickness: 2px;")
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "14pt")]},
        {"selector": "td", "props": [("font-size", "14pt")]},
    ])
)

,GPT5-mini,GPT5.2,Kimi
Characters,3.23,3.44,2.88
Coherence,3.92,4.25,3.98
Factuality,3.40,3.45,2.98
Fluency,3.51,4.02,3.74
Place,3.52,3.70,3.34
Plot,2.77,3.17,2.68
Relevance,2.88,3.22,3.06
Themes,3.07,2.92,2.68
Time,3.19,3.62,3.22


In [ ]:
# mean score across all categories and strategies

overall_model_scores = (
    unit_scores
    .groupby("model")["score"]
    .mean()
    .sort_values(ascending=False)
)

overall_model_scores

,score
model,
GPT5.2,3.531481
GPT5-mini,3.276852
Kimi,3.172222


**Performance without the metadata-only strategy**

In [ ]:
# now the metadata-only strategy is removed

df = combined_annotations_all_df.copy()

df = df[df["strategy"] != "Metadata"]

unit_scores_no_metadata = (
    df.groupby(
        ["novel_number", "summary_number", "model", "strategy", "category"]
    )["score"]
    .mean()
    .reset_index()
)


In [ ]:
model_category_table = unit_scores_no_metadata.pivot_table(
    index="category",
    columns="model",
    values="score",
    aggfunc="mean"
).round(2)

In [ ]:
model_category_table.index.name = None
model_category_table.columns.name = None

display(
    model_category_table.style
    .format("{:.2f}")
    .highlight_max(axis=1, props="text-decoration: underline; text-decoration-thickness: 2px;")
    .set_table_styles([
        {"selector": "th", "props": [("font-size", "14pt")]},
        {"selector": "td", "props": [("font-size", "14pt")]},
    ])
)

,GPT5-mini,GPT5.2,Kimi
Characters,3.73,3.98,3.34
Coherence,3.72,4.17,3.79
Factuality,3.82,3.98,3.50
Fluency,3.38,3.89,3.49
Place,3.99,4.18,3.78
Plot,3.09,3.61,3.09
Relevance,3.17,3.60,3.57
Themes,3.30,3.21,2.94
Time,3.47,4.07,3.53


In [ ]:
# mean score and standard deviation across all categories and the strategies (excluding the metadata-only)
model_stats = (
    unit_scores_no_metadata
    .groupby("model")["score"]
    .agg(["mean", "std", "count"])
    .round(2)
    .sort_values("mean", ascending=False)
)
model_stats

,mean,std,count
model,,,
GPT5.2,3.85,0.80,270
GPT5-mini,3.52,0.76,270
Kimi,3.45,0.78,270


In [ ]:
# standard deviation without metadata

model_task_std_no_metadata = unit_scores_no_metadata.pivot_table(
    index="category",
    columns="model",
    values="score",
    aggfunc="std"
).round(2)

model_task_std_no_metadata

model,GPT5-mini,GPT5.2,Kimi
category,,,
Characters,0.78,0.69,0.54
Coherence,0.48,0.59,0.64
Factuality,0.44,0.59,0.62
Fluency,0.59,0.60,0.74
Place,0.80,0.79,0.66
Plot,0.66,0.76,0.81
Relevance,0.59,0.76,0.74
Themes,0.81,0.83,0.92
Time,1.07,1.06,0.92


## Performance strategy

In [ ]:
# the summarization strategies.

df = combined_annotations_all_df.copy()

df["score"] = pd.to_numeric(df["score"], errors="coerce")

unit_scores = (
    df.groupby(
        ["novel_number", "summary_number", "strategy", "category"]
    )["score"]
    .mean()
    .reset_index()
)

In [ ]:
strategy_category_table = unit_scores.pivot_table(
    index="category",
    columns="strategy",
    values="score",
    aggfunc="mean"
).round(2)

In [ ]:
# Remember that the sequential approach is referred to as incremental updating in the paper.

strategy_category_table.index.name = None
strategy_category_table.columns.name = None

display(
    strategy_category_table.style
    .format("{:.2f}")
    .highlight_max(axis=1, props="text-decoration: underline;")
    .set_properties(**{"text-align": "center"})
)

,Full_text,Hierarchical,Metadata,Sequential
Characters,3.74,3.53,1.68,3.78
Coherence,4.09,4.04,4.53,3.54
Factuality,3.94,3.83,1.80,3.52
Fluency,3.78,3.68,4.27,3.30
Place,4.08,3.96,2.13,3.91
Plot,3.21,3.28,1.69,3.30
Relevance,3.52,3.49,1.88,3.32
Themes,3.29,3.30,2.09,2.87
Time,3.78,3.80,2.32,3.49


In [ ]:
# score across all models

overall_strategy_scores = (
    unit_scores
    .groupby("strategy")["score"]
    .mean()
    .sort_values(ascending=False)
)

overall_strategy_scores

,score
strategy,
Full_text,3.714815
Hierarchical,3.656790
Sequential,3.448148
Metadata,2.487654


In [ ]:
df = combined_annotations_all_df.copy()

df["score"] = pd.to_numeric(df["score"], errors="coerce")

unit_scores = (
    df.groupby(
        ["novel_number", "summary_number", "model", "strategy", "category"]
    )["score"]
    .mean()
    .reset_index()
)

In [ ]:
model_strategy_table = unit_scores.pivot_table(
    index="model",
    columns="strategy",
    values="score",
    aggfunc="mean"
).round(2)

In [ ]:
# Favorite strategy of the models

model_strategy_table.index.name = None
model_strategy_table.columns.name = None

display(
    model_strategy_table.style
    .format("{:.2f}")
    .highlight_max(axis=1, props="text-decoration: underline;")
    .set_properties(**{"text-align": "center"})
)

,Full_text,Hierarchical,Metadata,Sequential
GPT5-mini,3.57,3.42,2.55,3.57
GPT5.2,4.08,3.82,2.57,3.66
Kimi,3.49,3.73,2.34,3.12


In [ ]:
# mean and standard deviation of the strategies across all models

strategy_stats = (
    unit_scores
    .groupby("strategy")["score"]
    .agg(["mean", "std", "count"])
    .round(2)
    .sort_values("mean", ascending=False)
)
strategy_stats

,mean,std,count
strategy,,,
Full_text,3.71,0.83,270
Hierarchical,3.66,0.71,270
Sequential,3.45,0.83,270
Metadata,2.49,1.51,270


In [ ]:
# full text results for all models

category_model_fulltext = (
    unit_scores[unit_scores["strategy"] == "Full_text"]
    .pivot_table(
        index="category",
        columns="model",
        values="score",
        aggfunc="mean"
    )
    .round(2)
)

category_model_fulltext

model,GPT5-mini,GPT5.2,Kimi
category,,,
Characters,3.80,4.23,3.20
Coherence,3.83,4.60,3.83
Factuality,3.90,4.23,3.70
Fluency,3.33,4.00,4.00
Place,4.10,4.33,3.80
Plot,2.90,4.00,2.73
Relevance,3.20,3.97,3.40
Themes,3.57,3.10,3.20
Time,3.50,4.27,3.57


In [ ]:
# hierarchical merging results for all models

category_model_hierarchical = (
    unit_scores[unit_scores["strategy"] == "Hierarchical"]
    .pivot_table(
        index="category",
        columns="model",
        values="score",
        aggfunc="mean"
    )
    .round(2)
)

category_model_hierarchical

model,GPT5-mini,GPT5.2,Kimi
category,,,
Characters,3.37,3.73,3.50
Coherence,3.80,4.10,4.23
Factuality,3.90,3.93,3.67
Fluency,3.57,3.97,3.50
Place,4.13,4.00,3.73
Plot,2.83,3.40,3.60
Relevance,3.03,3.43,4.00
Themes,2.93,3.37,3.60
Time,3.20,4.47,3.73


In [ ]:
# incremental updating results for all models

category_model_sequential = (
    unit_scores[unit_scores["strategy"] == "Sequential"]
    .pivot_table(
        index="category",
        columns="model",
        values="score",
        aggfunc="mean"
    )
    .round(2)
)

category_model_sequential

model,GPT5-mini,GPT5.2,Kimi
category,,,
Characters,4.03,3.97,3.33
Coherence,3.53,3.80,3.30
Factuality,3.67,3.77,3.13
Fluency,3.23,3.70,2.97
Place,3.73,4.20,3.80
Plot,3.53,3.43,2.93
Relevance,3.27,3.40,3.30
Themes,3.40,3.17,2.03
Time,3.70,3.47,3.30


In [ ]:
# metadata-only results for all models

category_model_metadata = (
    unit_scores[unit_scores["strategy"] == "Metadata"]
    .pivot_table(
        index="category",
        columns="model",
        values="score",
        aggfunc="mean"
    )
    .round(2)
)

category_model_metadata

model,GPT5-mini,GPT5.2,Kimi
category,,,
Characters,1.73,1.83,1.47
Coherence,4.53,4.50,4.57
Factuality,2.13,1.87,1.40
Fluency,3.90,4.40,4.50
Place,2.10,2.27,2.03
Plot,1.80,1.83,1.43
Relevance,2.03,2.07,1.53
Themes,2.37,2.03,1.87
Time,2.37,2.30,2.30


## Performance on different novels

In [ ]:
df = combined_annotations_all_df.copy()

df = df[df["strategy"] == "Metadata"]

unit_scores_only_metadata = (
    df.groupby(
        ["novel_number", "novel_id", "summary_number", "model", "strategy", "category"]
    )["score"]
    .mean()
    .reset_index()
)

model_novel_table = (
    unit_scores_only_metadata
    .sort_values("novel_number")
    .pivot_table(
        index=["novel_number", "novel_id"],
        columns="model",
        values="score",
        aggfunc="mean"
    )
    .round(2)
    .reset_index()
    .set_index("novel_id")
    .drop(columns="novel_number")
)

model_novel_table

In [ ]:
# metadata-only inspection

model_novel_table.index.name = None
model_novel_table.columns.name = None

display(
    model_novel_table.style
    .format("{:.2f}")
    .highlight_max(axis=1, props="text-decoration: underline;")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([
        {"selector": "tbody tr:nth-child(4)",
         "props": [("border-bottom", "5px solid white")]}
    ])
)

,GPT5-mini,GPT5.2,Kimi
Fru Marie Grubbe,2.89,3.85,2.89
Niels Lyhne,2.96,3.22,2.78
Tine,2.93,3.74,2.07
Sult,4.41,4.19,3.93
Naturalisterne,3.44,1.78,2.11
Pavo,1.89,1.78,1.67
Mira,1.78,1.89,1.89
Diana,1.89,1.89,2.00
Præsten,1.78,1.67,2.22
Skurken på Nørrebro,1.56,1.67,1.89


In [ ]:
# applying the Margarat Cohen-inspired categories of social status
unit_scores_grouped = unit_scores.copy()

def assign_group(n):
    if n <= 4:
        return "Canon"
    elif n <= 7:
        return "Decontextualized"
    else:
        return "Forgotten"

unit_scores_grouped["novel_group"] = unit_scores_grouped["novel_number"].apply(assign_group)

unit_scores_grouped[["novel_number", "novel_group"]].drop_duplicates().sort_values("novel_number")

In [ ]:
model_strategy_group = unit_scores_grouped.pivot_table(
    index=["novel_group", "model"],
    columns="strategy",
    values="score",
    aggfunc="mean"
).round(2)

In [ ]:
# social status, strategies and models

model_strategy_group.index.names = [None, None]
model_strategy_group.columns.name = None

display(
    model_strategy_group.style
    .format("{:.2f}")
    .highlight_max(axis=1, props="text-decoration: underline;")
    .set_properties(**{"text-align": "center"})
)